In [2]:
import pandas as pd

# ขั้นตอนที่ 1: ดึงและทำความสะอาดข้อมูลโรงงาน ปี 63-64
# อ่านไฟล์ CSV โดยข้าม 2 บรรทัดแรกที่เป็นหัวรายงาน
df_6364 = pd.read_csv("63-64.csv", header=2)

# เทคนิค ffill (Forward Fill): แก้ปัญหา "Merge Cell" จากไฟล์ Excel
# โดยก็อปปี้ชื่อจังหวัดจากบรรทัดด้านบนลงมาเติมในบรรทัดที่เป็นค่าว่าง (NaN) ให้เต็ม
df_6364['จังหวัด'] = df_6364['จังหวัด'].ffill()

# กรองเอาเฉพาะข้อมูลแถวที่เป็น "จำนวนโรงงาน" (ตัดแถวที่บอกมูลค่าเงินลงทุนทิ้งไป)
df_6364 = df_6364[df_6364['รายการ'] == 'จำนวนโรงงาน (โรงงาน)'].copy()
df_6364['จังหวัด'] = df_6364['จังหวัด'].str.strip() # ลบช่องว่างหน้า/หลังชื่อจังหวัด

# Data Cleaning: แปลงตัวเลขที่มีเครื่องหมายลูกน้ำ (,) ให้เป็นตัวเลขทางคณิตศาสตร์ (Float/Int)
# errors='coerce' คือถ้าเจอข้อความแปลกประหลาดที่แปลงไม่ได้ ให้บังคับเป็นค่าว่าง (NaN) ไปเลย
df_6364['2563'] = pd.to_numeric(df_6364['2563'].astype(str).str.replace(',', ''), errors='coerce')
df_6364['2564'] = pd.to_numeric(df_6364['2564'].astype(str).str.replace(',', ''), errors='coerce')

# เลือกเฉพาะคอลัมน์ที่ใช้ และเปลี่ยนชื่อคอลัมน์ปี พ.ศ. ให้เป็น ค.ศ.
df_6364 = df_6364[['จังหวัด', '2563', '2564']]
df_6364.rename(columns={'2563': 2020, '2564': 2021}, inplace=True)

# Data Filtering: คัดกรองบรรทัดที่เป็น "ผลรวมระดับภาคและประเทศ" ทิ้งไป ให้เหลือแค่ 77 จังหวัด
exclude_words = ['ทั่วราชอาณาจักร', 'ภาคกลาง', 'ภาคเหนือ', 'ภาคใต้', 'ภาคตะวันออก', 'ภาคตะวันตก', 'ภาคตะวันออกเฉียงเหนือ', 'รวม', 'รวมทั่วประเทศ']
df_6364 = df_6364[~df_6364['จังหวัด'].isin(exclude_words)]


# ดึงและทำความสะอาดข้อมูลโรงงาน ปี 65, 66, 67 (ฟอร์แมตใหม่)
dfs = [] # สร้าง List ว่างไว้เก็บ DataFrame ของแต่ละปี
for year, year_en in zip([65, 66, 67], [2022, 2023, 2024]):
    filename = f"25{year}.csv"
    df = pd.read_csv(filename, header=2)

    # เลือกเฉพาะคอลัมน์ชื่อจังหวัด และคอลัมน์ยอดโรงงาน (ซึ่งไฟล์ต้นฉบับตั้งชื่อมาแปลกๆ ว่า ' จำนวน .2')
    df = df[['จังหวัด', ' จำนวน .2']].copy()
    df.rename(columns={' จำนวน .2': year_en}, inplace=True)

    # ลบแถวที่ไม่มีชื่อจังหวัด และลบช่องว่าง
    df = df.dropna(subset=['จังหวัด'])
    df['จังหวัด'] = df['จังหวัด'].astype(str).str.strip()
    df = df[~df['จังหวัด'].isin(exclude_words)]

    # Edge Case Handling: ดักจับและลบบรรทัดที่หัวตาราง '(โรง)' หลุดปนมาในข้อมูล
    df = df[~df[year_en].astype(str).str.contains(r'\(โรง\)', na=False)]

    # Data Cleaning :
    # 1. เปลี่ยนเครื่องหมายขีด '-' (แปลว่าไม่มีโรงงาน) ให้เป็นเลข '0'
    # 2. ลบลูกน้ำ ',' ออก
    df[year_en] = df[year_en].astype(str).str.replace(r'-', '0', regex=False).str.replace(',', '').str.strip()

    # แปลงเป็นตัวเลข
    df[year_en] = pd.to_numeric(df[year_en], errors='coerce')
    dfs.append(df)


# ประกอบร่างข้อมูลโรงงาน 5 ปี (Data Merging)
df_fac = df_6364.copy()
# วนลูปนำข้อมูลปี 65, 66, 67 มาต่อกันทีละปี โดยใช้การ Join แบบ Outer (เผื่อมีจังหวัดไหนตกหล่น)
for df in dfs:
    df_fac = pd.merge(df_fac, df, on='จังหวัด', how='outer')

# เติมค่าว่าง (NaN) ที่เกิดจากการ Join ให้เป็นเลข 0 (แปลว่าปีนั้นไม่มีการตั้งโรงงานเพิ่ม)
df_fac.fillna(0, inplace=True)

# ขั้นตอนที่ 4: ขยายร่างเป็นรายเดือน (Data Broadcasting)
# ปัญหา: ข้อมูลโรงงานเป็นรายปี แต่เราต้องวิเคราะห์ร่วมกับฝุ่นที่เป็นรายเดือน
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
results = []

for idx, row in df_fac.iterrows():
    prov = row['จังหวัด']
    for year in [2020, 2021, 2022, 2023, 2024]:
        val = row[year]
        # ขยายร่างข้อมูล 1 ปี ให้ซ้ำกัน 12 เดือน
        for m in months:
            results.append({
                'Year': year,
                'Month': m,
                'Province': prov,
                'Factories': int(val)
            })

df_fac_monthly = pd.DataFrame(results)

# บันทึกผลลัพธ์พร้อมทำ Encoding ภาษาไทย
df_fac_monthly.to_csv('Cleaned_Factories_Monthly.csv', index=False, encoding='utf-8-sig')

print(f"จำนวนทั้งหมด {len(df_fac_monthly)} แถว")
display(df_fac_monthly.head(10))

จำนวนทั้งหมด 4620 แถว


,Year,Month,Province,Factories
0,2020,Jan,กระบี่,24
1,2020,Feb,กระบี่,24
2,2020,Mar,กระบี่,24
3,2020,Apr,กระบี่,24
4,2020,May,กระบี่,24
5,2020,Jun,กระบี่,24
6,2020,Jul,กระบี่,24
7,2020,Aug,กระบี่,24
8,2020,Sep,กระบี่,24
9,2020,Oct,กระบี่,24
